In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import plotly.express as px
import torch
import pandas as pd
import time
from tqdm import tqdm
from glob import glob
import numpy as np
import sqlite3
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm
import joblib
import platform
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio




/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

if platform.node().startswith("jupyter") or "cluster" in platform.node().lower():
    # on cluster
    extracted_dir = os.path.expanduser("~/telegram_2024/usc-tg-24-us-election/extracted")
else:
    # on laptop
    extracted_dir = os.path.join("..", "material", "extracted")

print("✅ Using path:", extracted_dir)


✅ Using path: /home/students/s328743/telegram_2024/usc-tg-24-us-election/extracted


In [3]:
chats_path = '../material/chats.db'
conn = sqlite3.connect(chats_path)
cursor=conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables=cursor.fetchall()

# ========================
# 1. Leggi chats.db (SQLite)
# ========================

print("Tables in DB:", tables)
try:
    df_chats = pd.read_sql_query("SELECT * FROM chats", conn)
    df_chats =df_chats.drop_duplicates(subset='type_and_id')
    print("number of unique chats", len(df_chats))
    print("chats.db - Tabella 'chats'")
    print(df_chats.head())
except Exception as e:
    print("Errore nel leggere la tabella:", e)

conn.close()

# ========================
# 2. Leggi discovery_edges.csv.gz
# ========================
try:
    df_edges = pd.read_csv('../material/discovery_edges.csv.gz')
    df_edges = df_edges.drop_duplicates(subset='type_and_id')
    print("✅ discovery_edges.csv.gz, \n" \
    "Il timestamp da l'ultima volta che hanno visitato quel gruppo ma questo significa che non è davvero indicativo di una timeline \n")
    print(df_edges.head())
except Exception as e:
    print("Errore nel leggere discovery_edges:", e)

# ========================
# 3. Leggi first_nodes.csv.gz
# ========================
try:
    df_first_nodes = pd.read_csv('../material/first_nodes.csv.gz')
    print("number of non unique first nodes", len(df_first_nodes))
    df_first_nodes = df_first_nodes.drop_duplicates(subset='type_and_id')
    print("✅ first_nodes.csv.gz")
    print(df_first_nodes.head())
    print("number of unique first nodes", len(df_first_nodes))
except Exception as e:
    print("Errore nel leggere first_nodes:", e)

print("type_and_id unique in df_first_nodes" + str(df_first_nodes.type_and_id.nunique()))
print("type_and_id in df_first_nodes" + str(len(df_first_nodes)))
print("type_and_id NaN in df_first_nodes " + str(df_first_nodes['type_and_id'].isna().sum()))

Tables in DB: [('chats',)]
number of unique chats 127141
chats.db - Tabella 'chats'
           type_and_id                   token                  parent  \
0                 None  [keyword] thedemocrats                    None   
40  channel_1889806290         thedemocratskmf  [keyword] thedemocrats   
41  channel_1413288788       thedemocratsindia  [keyword] thedemocrats   
42  channel_1709284265           thedemocratsa  [keyword] thedemocrats   
43  channel_1807294270             democratsmv  [keyword] thedemocrats   

       timestamp  
0   1.722583e+09  
40  1.728093e+09  
41  1.728093e+09  
42  1.728093e+09  
43  1.728093e+09  
✅ discovery_edges.csv.gz, 
Il timestamp da l'ultima volta che hanno visitato quel gruppo ma questo significa che non è davvero indicativo di una timeline 

          type_and_id              parent     timestamp
0  channel_1306559115  channel_1840578235  1.722586e+09
1  channel_2036850729  channel_1840578235  1.722586e+09
2  channel_1941222046  channel_18

In [4]:
df_preprocessed_non_empty_channels = pd.read_csv(
    "../material/preprocessed_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_preprocessed_non_empty_channels_only_with_short_messages = pd.read_csv(
    "../material/preprocessed_short_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_preprocessed_non_empty_channels_spam_messages = pd.read_csv(
    "../material/preprocessed_spam_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_channels_without_message = pd.read_csv(
    "../material/channels_without_message.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_english_preprocessed_non_empty_channels = pd.read_csv(
    "../material/preprocessed_english_messages.tsv.gz", 
    sep='\t', 
    compression='gzip'
)

df_sampled = pd.read_csv(
    "../final/df_sampled.csv"
)

print("--- len of df_sampled", len(df_sampled))
print("--- first rows of df_channels_without_message", df_channels_without_message.head(1))
print("--- first rows of df_preprocessed_non_empty_channels", df_preprocessed_non_empty_channels.head(1))
print("--- first rows of df_english_preprocessed_non_empty_channels:", df_english_preprocessed_non_empty_channels.head(1))
print("\n---\n--- number of initial seeds:", str(len(df_first_nodes)))
print("--- Unique channels in df_channels_without_message:", df_channels_without_message['channel_id'].nunique())
print("--- Number of distinct channel_ids in file_args: 180")
print("--- Distinct channels: Unique channel ids with messages in df_preprocessed_non_empty_channels(also before modification)", df_preprocessed_non_empty_channels['channel_id'].nunique())
print("--- Distinct english channels: Unique channels in df_english_preprocessed_non_empty_channels:", df_english_preprocessed_non_empty_channels['channel_id'].nunique())


--- len of df_sampled 154346
--- first rows of df_channels_without_message            channel_id
0  channel_1413288788
--- first rows of df_preprocessed_non_empty_channels                                                 text   timestamp  \
0  Welcome to the Republican Party $GOP where we ...  1719544548   

                                   text_preprocessed language  \
0  welcome republican party proudly loudly suppor...       en   

           channel_id               token                 date  
0  channel_2202860593  republicanpartyeth  2024-06-28 03:15:48  
--- first rows of df_english_preprocessed_non_empty_channels:                                                 text   timestamp  \
0  Welcome to the Republican Party $GOP where we ...  1719544548   

                                   text_preprocessed language  \
0  welcome republican party proudly loudly suppor...       en   

           channel_id               token                 date  
0  channel_2202860593  republicanpa

- reduce outliers